# iRage AlgoArena — Short-Horizon Return Prediction
## Final Submission Notebook (v9)

**v9 vs v8 — Why the rewrite:**

| Component | v8 (local) | v8 (Kaggle 2.5×) | v9 (Kaggle est.) |
|---|---|---|---|
| Feature selection (LGB pass) | 193s | **476s** | **~5s** (corr-based) |
| CatBoost 5 folds | 376s | ~940s | **dropped** |
| LGB 5 folds | 683s | ~1707s | ~630s (3 folds, faster LR) |
| **Total** | **1699s** | **~4200s** | **~825s** |

**Key decisions:**
- Feature selection: Pearson |corr(X,y)| via numpy matmul — zero model training, same ranking quality
- 3-fold GroupKFold instead of 5 (valid CV, saves 40% model time)
- Dropped CatBoost (15 min alone on Kaggle at 2.5× slowdown)
- LGB learning rate 0.02→0.05 (Huber), 0.05→0.08 (Shallow)
- 150 features instead of 300 (faster per fold, corr filter is tighter)
- Hard stop at 25 min (1500s)

**Compliance:**
- Data: `/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage/`
- No internet, no pip, no external data
- All FE is row-wise only
- Normalization + selection use train data only
- GroupKFold on CV_GROUP

In [ ]:
# CELL 1 - IMPORTS
import os, gc, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize

import lightgbm as lgb
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import r2_score

SEED = 23
np.random.seed(SEED)
NOTEBOOK_START = time.time()

print(f'LightGBM {lgb.__version__}')
print('Imports OK')

# ============================================================

In [ ]:
# CELL 2 - CONFIG
# ============================================================
# v9 ARCHITECTURE DECISIONS:
#
# [1] NO model-based feature selection
#     v8 used a 500-round LGB pass on the full 916-feature set.
#     On Kaggle this alone took 476s (8 min of the 30 min budget).
#     v9 uses pure numpy: variance filter + target correlation.
#     Cost: ~5-10s. Zero model training required.
#     Quality: correlation-based selection is actually MORE
#     aligned with the R2 objective than gain importance.
#
# [2] 3 folds instead of 5
#     5 folds × 2 models = 10 training runs on 530k rows each.
#     3 folds × 2 models = 6 training runs on 440k rows each.
#     Saves 40% of model training time. OOF estimates are
#     noisier but still valid — the competition evaluates on
#     test predictions, not OOF R2.
#
# [3] Drop CatBoost
#     CAT took 376s locally → ~940s on Kaggle = 15 min alone.
#     OOF R2 was 0.001451 vs LGB_Huber 0.001227 (small margin).
#     Not worth 15 min when LGB_Huber is almost as good.
#     Pure LGB ensemble is faster, more stable, and easier to
#     time-budget precisely.
#
# [4] Fewer engineered features: 471 → 220
#     Skip family aggregates (20 features, expensive .std() calls)
#     Skip SO3_T poly interactions (7 features, marginal gain)
#     Keep: momentum (330), absmom (110), price (4) → ~444 new
#     Wait — we keep ALL FE but select 150 instead of 300.
#     FE is fast (42s local). Selection was the bottleneck.
#
# [5] Selection: top 150 by |corr(feature, target)|
#     After z-score, compute abs pearson correlation of each
#     feature with target using vectorized numpy matmul.
#     661k × 916 matmul: ~3s. No model training needed.
#     150 features is enough — v8 showed zero-importance on 82/916.
#
# [6] Faster LGB params
#     learning_rate: 0.02 → 0.05 (2.5x faster convergence)
#     num_boost_round: 5000 → 2000 (hard cap, early stop still active)
#     This matches the effective budget we have per fold.

class CFG:
    DATA_DIR   = Path('/kaggle/input/competitions/short-horizon-return-prediction-challenge-by-i-rage')
    TRAIN_PATH = DATA_DIR / 'train.parquet'
    TEST_PATH  = DATA_DIR / 'test.parquet'
    OUTPUT_DIR = Path('/kaggle/working')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    N_FOLDS = 3          # down from 5: saves 40% model time
    SEED    = 23

    CLIP_LOWER = 2       # P2/P98 confirmed better in v5/v6 comparison
    CLIP_UPPER = 98

    # Feature engineering flags (all row-wise, compliance-safe)
    BUILD_MOMENTUM     = True
    BUILD_ABS_MOMENTUM = True
    BUILD_PRICE_FEATS  = True
    BUILD_FAMILY_AGG   = True   # keep — fast, only 20 feats
    BUILD_SO3_POLY     = True   # keep — fast, only 7 feats

    # Selection: top N by |corr(X, y)| — pure numpy, ~5s
    N_TOP_FEATURES = 150

    STANDARDIZE = True

    # ── LGB_Huber: moderate depth, Huber loss ──────────────
    # lr=0.05: faster than v8's 0.02, still finds patterns.
    # num_leaves=15: same as v8 (regime memorization fix).
    # early_stop=100: generous enough to not cut short.
    LGB_HUBER_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.05,       # v9: 2.5x faster than v8
        'num_leaves':        15,
        'max_depth':         -1,
        'min_child_samples': 200,
        'feature_fraction':  0.6,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         0.5,
        'lambda_l2':         8.0,
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED,
    }

    # ── LGB_Shallow: very few leaves, near-global patterns ─
    # lr=0.08: even faster, fewer useful patterns at 7 leaves.
    # early_stop=60: shallow models plateau fast.
    LGB_SHALLOW_PARAMS = {
        'objective':         'huber',
        'alpha':             0.02,
        'metric':            'huber',
        'learning_rate':     0.08,       # v9: faster
        'num_leaves':        7,
        'max_depth':         -1,
        'min_child_samples': 500,
        'feature_fraction':  0.5,
        'bagging_fraction':  0.8,
        'bagging_freq':      1,
        'lambda_l1':         1.0,
        'lambda_l2':         15.0,
        'min_gain_to_split': 0.0,
        'verbose':           -1,
        'n_jobs':            -1,
        'seed':              SEED + 2,
    }

    LGB_NUM_ROUNDS  = 2000   # hard cap (early stop will fire first)
    LGB_EARLY_STOP  = 100    # generous
    SHAL_EARLY_STOP = 60

    # Hard stop: 25 min = 1500s (gives 5 min margin to Kaggle's 30)
    RUNTIME_LIMIT_S = 1500

print('Config OK')

# ============================================================

In [ ]:
# CELL 3 - DATA LOADING
# ============================================================
print('Loading data...')
t0 = time.time()

train = pd.read_parquet(CFG.TRAIN_PATH)
test  = pd.read_parquet(CFG.TEST_PATH)

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Loaded in {time.time()-t0:.1f}s')

train_ids = train['ID'].values.copy()
test_ids  = test['ID'].values.copy()
y_raw     = train['TARGET'].copy()

if 'CV_GROUP' in train.columns:
    cv_groups    = train['CV_GROUP'].values.copy()
    USE_GROUP_CV = True
    n_groups     = len(np.unique(cv_groups))
    print(f'CV_GROUP: {n_groups} groups -> GroupKFold({CFG.N_FOLDS})')
    train.drop(columns=['ID', 'TARGET', 'CV_GROUP'], inplace=True)
else:
    cv_groups    = None
    USE_GROUP_CV = False
    print('No CV_GROUP -> KFold fallback')
    train.drop(columns=['ID', 'TARGET'], inplace=True)

test.drop(columns=['ID'], inplace=True)

assert list(train.columns) == list(test.columns), 'FATAL: column mismatch!'
print(f'Features: {train.shape[1]} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

# ============================================================

In [ ]:
# CELL 4 - TARGET CLIPPING
# ============================================================
desc = y_raw.describe(percentiles=[0.01, 0.02, 0.05, 0.25, 0.5, 0.75, 0.95, 0.98, 0.99])
print(desc)
print(f'Kurtosis: {y_raw.kurtosis():.2f}')

clip_lo  = y_raw.quantile(CFG.CLIP_LOWER / 100)
clip_hi  = y_raw.quantile(CFG.CLIP_UPPER / 100)
y_series = y_raw.clip(clip_lo, clip_hi)
n_clip   = int(((y_raw < clip_lo) | (y_raw > clip_hi)).sum())
print(f'Clip P{CFG.CLIP_LOWER}/P{CFG.CLIP_UPPER}: [{clip_lo:.6f}, {clip_hi:.6f}]')
print(f'Clipped: {n_clip:,} rows ({100*n_clip/len(y_raw):.2f}%) | Std: {y_raw.std():.4f} -> {y_series.std():.4f}')

y = y_series.values.astype(np.float32)
print('Target ready OK')

# ============================================================

In [ ]:
# CELL 5 - FEATURE CATEGORIZATION
# ============================================================
all_cols  = train.columns.tolist()
lag1_cols = [c for c in all_cols if c.endswith('_LagT1')]
lag2_cols = [c for c in all_cols if c.endswith('_LagT2')]
lag3_cols = [c for c in all_cols if c.endswith('_LagT3')]
lag_set   = set(lag1_cols + lag2_cols + lag3_cols)
base_cols = [c for c in all_cols if c not in lag_set]

special_cols   = ['Price', 'SO3_T']
pure_base_cols = [c for c in base_cols if c not in special_cols]

def get_family(col):
    return col.split('_')[0]

families = sorted(set(get_family(c) for c in pure_base_cols))

lag_map = {
    c: {
        'lag1': c+'_LagT1' if c+'_LagT1' in lag_set else None,
        'lag2': c+'_LagT2' if c+'_LagT2' in lag_set else None,
        'lag3': c+'_LagT3' if c+'_LagT3' in lag_set else None,
    } for c in pure_base_cols
}

print(f'Pure base: {len(pure_base_cols)} | Lag1/2/3: {len(lag1_cols)}/{len(lag2_cols)}/{len(lag3_cols)}')
print(f'Families: {families} | Total: {len(all_cols)}')

# ============================================================

In [ ]:
# CELL 6 - FEATURE ENGINEERING (row-wise only, compliance-safe)
# ============================================================
# ALL operations: purely element-wise or row-wise aggregations.
# NO cross-sample operations. Identical function on train/test.

def feature_engineering(df, lag_map, families, pure_base_cols, verbose=True):
    df = df.copy()
    new_feats = []

    # A. MOMENTUM: acceleration and divergence signals
    if CFG.BUILD_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1, l2, l3 = lags['lag1'], lags['lag2'], lags['lag3']
            if l1 and l2 and l1 in df.columns and l2 in df.columns:
                df[bc+'_Acc12'] = df[l1] - df[l2]; new_feats.append(bc+'_Acc12')
            if l2 and l3 and l2 in df.columns and l3 in df.columns:
                df[bc+'_Acc23'] = df[l2] - df[l3]; new_feats.append(bc+'_Acc23')
            if l1 and l3 and l1 in df.columns and l3 in df.columns:
                df[bc+'_Div13'] = df[l1] - df[l3]; new_feats.append(bc+'_Div13')
        if verbose: print(f'  Momentum: {len(new_feats)-n0}')

    # B. |LagT1|: momentum magnitude
    if CFG.BUILD_ABS_MOMENTUM:
        n0 = len(new_feats)
        for bc, lags in lag_map.items():
            l1 = lags['lag1']
            if l1 and l1 in df.columns:
                df[bc+'_AbsMom'] = df[l1].abs(); new_feats.append(bc+'_AbsMom')
        if verbose: print(f'  AbsMom:   {len(new_feats)-n0}')

    # C. Price features
    if CFG.BUILD_PRICE_FEATS and 'Price' in df.columns:
        n0 = len(new_feats)
        pl1, pl2, pl3 = 'Price_LagT1', 'Price_LagT2', 'Price_LagT3'
        if pl1 in df.columns and pl2 in df.columns:
            df['Price_Acc']    = df[pl1] - df[pl2]; new_feats.append('Price_Acc')
        if pl1 in df.columns and pl3 in df.columns:
            df['Price_Div13']  = df[pl1] - df[pl3]; new_feats.append('Price_Div13')
        if pl2 in df.columns and pl3 in df.columns:
            df['Price_Acc23']  = df[pl2] - df[pl3]; new_feats.append('Price_Acc23')
        if pl1 in df.columns:
            df['Price_AbsMom'] = df[pl1].abs();      new_feats.append('Price_AbsMom')
        if verbose: print(f'  Price:    {len(new_feats)-n0}')

    # D. Family aggregates
    if CFG.BUILD_FAMILY_AGG:
        n0 = len(new_feats)
        for fam in families:
            fb  = [c for c in pure_base_cols if get_family(c)==fam and c in df.columns]
            fl1 = [c+'_LagT1' for c in fb if c+'_LagT1' in df.columns]
            if len(fb) < 2: continue
            df[fam+'_BaseMean']  = df[fb].mean(axis=1);  new_feats.append(fam+'_BaseMean')
            df[fam+'_BaseStd']   = df[fb].std(axis=1);   new_feats.append(fam+'_BaseStd')
            if len(fl1) >= 2:
                fmom = df[fl1]
                df[fam+'_MomMean']   = fmom.mean(axis=1); new_feats.append(fam+'_MomMean')
                df[fam+'_MomStd']    = fmom.std(axis=1);  new_feats.append(fam+'_MomStd')
                df[fam+'_MomSpread'] = fmom.max(axis=1) - fmom.min(axis=1); new_feats.append(fam+'_MomSpread')
        if verbose: print(f'  FamilyAgg:{len(new_feats)-n0}')

    # E. SO3_T polynomial
    if CFG.BUILD_SO3_POLY and 'SO3_T' in df.columns:
        n0 = len(new_feats)
        df['SO3_T_sq'] = df['SO3_T']**2; new_feats.append('SO3_T_sq')
        df['SO3_T_cb'] = df['SO3_T']**3; new_feats.append('SO3_T_cb')
        if 'Price_LagT1' in df.columns:
            df['SO3T_x_PriceMom'] = df['SO3_T'] * df['Price_LagT1']
            new_feats.append('SO3T_x_PriceMom')
        for fam in families:
            fn = fam+'_MomMean'
            if fn in df.columns:
                df['SO3T_x_'+fam+'_Mom'] = df['SO3_T'] * df[fn]
                new_feats.append('SO3T_x_'+fam+'_Mom')
        if verbose: print(f'  SO3_T:    {len(new_feats)-n0}')

    if verbose: print(f'  Total new: {len(new_feats)} | Grand total: {df.shape[1]}')
    return df, new_feats

print('Running FE...')
t0 = time.time()
train_fe, new_feats = feature_engineering(train, lag_map, families, pure_base_cols)
test_fe,  _         = feature_engineering(test,  lag_map, families, pure_base_cols, verbose=False)
print(f'Done in {time.time()-t0:.1f}s | Train: {train_fe.shape} | Test: {test_fe.shape}')

# ============================================================

In [ ]:
# CELL 7 - FINALIZE ARRAYS
# ============================================================
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace( [np.inf, -np.inf], np.nan, inplace=True)
assert list(train_fe.columns) == list(test_fe.columns), 'FATAL: column mismatch!'

feature_cols = train_fe.columns.tolist()
n_features   = len(feature_cols)
print(f'Total features: {n_features}')

X      = train_fe.values.astype(np.float32)
X_test = test_fe.values.astype(np.float32)
print(f'X: {X.shape} | X_test: {X_test.shape}')

del train_fe, test_fe, train, test
gc.collect()
print(f'Memory freed | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

# ============================================================

In [ ]:
# CELL 8 - Z-SCORE STANDARDIZATION (TRAIN STATS ONLY)
# ============================================================
# Must happen BEFORE feature selection so correlations are
# computed on the same scale as the model will train on.
# Train stats only — no test information used.

if CFG.STANDARDIZE:
    t_std   = time.time()
    tr_mean = np.nanmean(X, axis=0).astype(np.float32)
    tr_std  = np.nanstd(X,  axis=0).astype(np.float32)
    tr_std  = np.where(tr_std < 1e-8, 1.0, tr_std)

    X      = np.clip((X      - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X_test = np.clip((X_test - tr_mean) / tr_std, -10, 10).astype(np.float32)
    X      = np.nan_to_num(X,      nan=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0)
    print(f'Standardized | X mean={X.mean():.4f} std={X.std():.4f} | {time.time()-t_std:.1f}s')

print(f'Arrays: X={X.shape} X_test={X_test.shape} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

# ============================================================

In [ ]:
# CELL 9 - FEATURE SELECTION (ZERO-MODEL — PURE NUMPY)
# ============================================================
# v8 bottleneck: trained a 500-round LGB model = 193s local,
# 476s on Kaggle. This burned 8 of the 30 minute budget.
#
# v9 fix: vectorized Pearson correlation via numpy.
# |corr(X_i, y)| = |dot(X_i, y)| / n  (after z-scoring both).
# Total cost: one matrix-vector multiply = ~3-5 seconds.
#
# Why this works for R2 optimization:
#   R2 = 1 - SS_res/SS_tot
#   A feature's marginal R2 contribution is exactly its squared
#   Pearson correlation with the residual. At initialization
#   (residual = y - mean(y)), the best features are those most
#   correlated with y. This is a tighter proxy than LGB gain,
#   which reflects tree-structure complexity not predictive value.
#
# Additional filter: near-zero variance features are removed first
# (already caught by tr_std < 1e-8 in standardization, but
# explicit filter adds a second safety net).
#
# COMPLIANCE: uses ONLY training data (X and y). No test info.

print(f'Feature selection: {len(feature_cols)} -> top {CFG.N_TOP_FEATURES} by |corr(X,y)|...')
t_fs = time.time()

# y must be centered (for Pearson). y_centered = y - mean(y)
y_c = (y - y.mean()).astype(np.float32)
y_norm = np.sqrt((y_c**2).sum())   # ||y_c||

# Vectorized: X is already z-scored (mean~0, std~1)
# corr_i = dot(X[:,i], y_c) / (n * std_X_i * std_y)
# But X is z-scored so std_X_i ≈ 1, so corr_i ≈ dot(X[:,i], y_c) / (n * std_y)
# We only need ranking, so we compute dot products only:
# Shape: (n_features,)
dots = X.T @ y_c      # (F,) — fast BLAS matmul

# Normalize by ||X_i|| * ||y_c|| for proper Pearson correlation
X_norms = np.sqrt((X**2).sum(axis=0))   # (F,)
X_norms = np.where(X_norms < 1e-8, 1.0, X_norms)
corr    = np.abs(dots / (X_norms * y_norm))  # absolute Pearson corr (F,)

# Rank and select
rank_idx  = np.argsort(corr)[::-1]              # descending
sel_idx   = rank_idx[:CFG.N_TOP_FEATURES]
sel_feats = [feature_cols[i] for i in sel_idx]

# Sort by index to preserve original column order (speeds up LGB)
sel_idx_sorted   = np.sort(sel_idx)
sel_feats_sorted = [feature_cols[i] for i in sel_idx_sorted]

X          = X[:, sel_idx_sorted]
X_test     = X_test[:, sel_idx_sorted]
feature_cols = sel_feats_sorted
n_features   = len(feature_cols)

top5_corr = [(feature_cols[i] if i < len(feature_cols) else '?', corr[sel_idx[j]])
             for j, i in enumerate(sel_idx[:5])]

print(f'Zero-model selection done in {time.time()-t_fs:.1f}s')
print(f'Selected: {n_features} features')
print(f'Top 5 by |corr|: {[(f, f"{c:.6f}") for f, c in top5_corr]}')
print(f'Final arrays: X={X.shape} X_test={X_test.shape} | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

gc.collect()

# ============================================================

In [ ]:
# CELL 10 - CV SPLITTER
# ============================================================
if USE_GROUP_CV:
    splitter = GroupKFold(n_splits=CFG.N_FOLDS)
    splits   = list(splitter.split(X, y, groups=cv_groups))
    print(f'GroupKFold({CFG.N_FOLDS}) on CV_GROUP ({n_groups} groups)')
    for i, (ti, vi) in enumerate(splits):
        print(f'  Fold {i+1}: {len(ti):,} train | {len(vi):,} val | {len(np.unique(cv_groups[vi]))} groups')
else:
    splits = list(KFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(X, y))
    print('KFold fallback')
print('CV ready OK')

# ============================================================

In [ ]:
# CELL 11 - GENERIC LGB TRAINER
# ============================================================
def run_lgb_cv(params, num_rounds, early_stop, splits, X, y, X_test,
               feature_cols, name='LGB'):
    oof   = np.zeros(len(y),       dtype=np.float64)
    ptest = np.zeros(len(X_test),  dtype=np.float64)
    scores, iters = [], []
    n_folds_run = 0

    print(f'--- {name} ({len(splits)} folds) ---')
    for fold, (ti, vi) in enumerate(splits):
        elapsed = time.time() - NOTEBOOK_START
        if elapsed > CFG.RUNTIME_LIMIT_S * 0.85:
            print(f'  Time budget ({elapsed:.0f}s): stopping {name} at fold {fold}')
            break

        t0   = time.time()
        dtr  = lgb.Dataset(X[ti], label=y[ti], feature_name=feature_cols, free_raw_data=True)
        dval = lgb.Dataset(X[vi], label=y[vi], reference=dtr,             free_raw_data=True)

        m = lgb.train(
            params, dtr,
            num_boost_round=num_rounds,
            valid_sets=[dtr, dval],
            valid_names=['train', 'valid'],
            callbacks=[
                lgb.early_stopping(early_stop, verbose=False),
                lgb.log_evaluation(500),
            ],
        )
        bi = m.best_iteration
        iters.append(bi)

        vp        = m.predict(X[vi],    num_iteration=bi)
        oof[vi]   = vp
        ptest    += m.predict(X_test,   num_iteration=bi) / len(splits)

        fr2 = r2_score(y[vi], vp)
        scores.append(fr2)
        n_folds_run += 1
        print(f'  Fold {fold+1}/{len(splits)} | R2: {fr2:+.6f} | iter: {bi} | {time.time()-t0:.0f}s')

        del m, dtr, dval
        gc.collect()

    if n_folds_run == 0:
        print(f'  WARNING: {name} — no folds completed')
        return oof, ptest, iters, scores, 0.0

    if n_folds_run < len(splits):
        # Rescale test predictions for the completed folds
        ptest = ptest * len(splits) / n_folds_run
        fi    = np.concatenate([splits[i][1] for i in range(n_folds_run)])
        oof_r2 = r2_score(y[fi], oof[fi])
    else:
        oof_r2 = r2_score(y, oof)

    print(f'  {name} OOF R2: {oof_r2:+.6f} | std: {np.std(scores):.6f} | avg_iter: {np.mean(iters):.0f}')
    return oof, ptest, iters, scores, oof_r2

print('LGB trainer defined OK')

# ============================================================

In [ ]:
# CELL 12 - RUN LGB MODELS
# ============================================================
print('='*55)
print(f'LightGBM CV — {CFG.N_FOLDS} folds x 2 models')
print('v9: no CatBoost, 3-fold, faster LR, corr-based selection')
print('='*55)

oof_lgb_huber, pred_lgb_huber, iters_huber, scores_huber, r2_lgb_huber = run_lgb_cv(
    CFG.LGB_HUBER_PARAMS, CFG.LGB_NUM_ROUNDS, CFG.LGB_EARLY_STOP,
    splits, X, y, X_test, feature_cols, name='LGB_Huber'
)

oof_lgb_shal, pred_lgb_shal, iters_shal, scores_shal, r2_lgb_shal = run_lgb_cv(
    CFG.LGB_SHALLOW_PARAMS, CFG.LGB_NUM_ROUNDS, CFG.SHAL_EARLY_STOP,
    splits, X, y, X_test, feature_cols, name='LGB_Shallow'
)

print(f'Both models done | Elapsed: {time.time()-NOTEBOOK_START:.0f}s')

# ============================================================

In [ ]:
# CELL 13 - ENSEMBLE
# ============================================================
model_names = ['LGB_Huber', 'LGB_Shallow']
oofs_list   = [oof_lgb_huber, oof_lgb_shal]
preds_list  = [pred_lgb_huber, pred_lgb_shal]

print('ENSEMBLE OPTIMIZATION')
print('-'*50)
for nm, oof in zip(model_names, oofs_list):
    print(f'  {nm:<14} OOF R2: {r2_score(y, oof):+.6f}')

oof_sa  = np.mean(oofs_list, axis=0)
pred_sa = np.mean(preds_list, axis=0)
r2_sa   = r2_score(y, oof_sa)
print(f'Simple avg:  {r2_sa:+.6f}')

oofs_arr  = np.stack(oofs_list, axis=1)
preds_arr = np.stack(preds_list, axis=1)

def neg_r2(rw):
    w = np.exp(rw) / np.exp(rw).sum()
    return -r2_score(y, (oofs_arr * w).sum(axis=1))

best_r2_opt, best_w = -np.inf, None
for s in range(10):
    rng = np.random.RandomState(s)
    w0  = rng.randn(len(oofs_list)) * 0.1
    res = minimize(neg_r2, w0, method='Nelder-Mead',
                   options={'maxiter': 2000, 'xatol': 1e-7, 'fatol': 1e-9})
    if -res.fun > best_r2_opt:
        best_r2_opt = -res.fun
        best_w      = np.exp(res.x) / np.exp(res.x).sum()

oof_opt  = (oofs_arr  * best_w).sum(axis=1)
pred_opt = (preds_arr * best_w).sum(axis=1)
r2_opt   = r2_score(y, oof_opt)

print(f'Optimized:   {r2_opt:+.6f}')
for nm, w in zip(model_names, best_w):
    print(f'  Weight {nm}: {w:.4f}')

if r2_opt > r2_sa:
    final_oof, final_pred = oof_opt, pred_opt
    final_r2, method_str  = r2_opt, 'Optimized wts'
else:
    final_oof, final_pred = oof_sa, pred_sa
    final_r2, method_str  = r2_sa, 'Simple avg'

print(f'FINAL: {final_r2:+.6f} ({method_str})')

# ============================================================

In [ ]:
# CELL 14 - SUBMISSION GUARDS + SAVE
# ============================================================
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'ID': test_ids, 'TARGET': final_pred})

assert len(submission)            == len(test_ids),  'FATAL: row count'
assert list(submission.columns)  == ['ID','TARGET'], 'FATAL: columns'
assert submission['ID'].nunique() == len(test_ids),  'FATAL: dup IDs'
assert not submission['TARGET'].isnull().any(),       'FATAL: NaN'
assert not np.isinf(submission['TARGET'].values).any(),'FATAL: Inf'
assert submission['TARGET'].std() > 0,               'FATAL: constant'

print('All 6 guards passed OK')

sub_path = CFG.OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

v = pd.read_csv(sub_path)
assert len(v) == len(test_ids)
assert list(v.columns) == ['ID', 'TARGET']

total_elapsed = time.time() - NOTEBOOK_START
print(f'submission.csv SAVED | {len(submission):,} rows | {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print(submission['TARGET'].describe())

# ============================================================

In [ ]:
# CELL 15 - SCORECARD
# ============================================================
total_elapsed = time.time() - NOTEBOOK_START

print('='*70)
print('  R2 SCORECARD — iRage Short-Horizon Return Prediction v9')
print('='*70)
print(f'  Features: {n_features} (selected from {X.shape[1] + n_features} by |corr|)')
print(f'  Train: 661,574 | Test: 410,139')
print(f'  CV: {"GroupKFold("+str(CFG.N_FOLDS)+")" if USE_GROUP_CV else "KFold("+str(CFG.N_FOLDS)+")"}')
print(f'  Runtime: {total_elapsed:.0f}s = {total_elapsed/60:.1f} min')
print('-'*70)
print('  MODEL OOF R2')
for nm, oof in zip(model_names, oofs_list):
    r2  = r2_score(y, oof)
    bar = '#' * min(50, max(0, int(r2 * 1000)))
    print(f'  {nm:<14} {r2:+.6f}  {bar}')
print('-'*70)
print('  ENSEMBLE')
print(f'  Simple avg:  {r2_sa:+.6f}')
print(f'  Optimized:   {r2_opt:+.6f}')
print(f'  FINAL ({method_str}): {final_r2:+.6f}')
print('-'*70)
print('  OVERFIT DIAGNOSTIC')
print(f'  LGB_Huber  avg best_iter: {np.mean(iters_huber):.0f}')
print(f'  LGB_Shallow avg best_iter: {np.mean(iters_shal):.0f}')
print('-'*70)
print('  v9 CHANGES vs v8')
print('  + Feature selection: LGB-based (476s Kaggle) -> corr-based (~5s)')
print('  + N folds: 5 -> 3 (saves 40% model time)')
print('  + No CatBoost (was 940s on Kaggle at 2.5x)')
print('  + LGB lr: 0.02 -> 0.05 (Huber), 0.05 -> 0.08 (Shallow)')
print('  + N features: 300 -> 150 (faster per-fold, less noise)')
print('  + Hard runtime limit: 1500s (25 min)')
print('-'*70)
print('  COMPLIANCE')
print('  OK Data: /kaggle/input/competitions/...')
print('  OK No internet / pip / external data')
print('  OK GroupKFold on CV_GROUP')
print('  OK Zero cross-sample test operations')
print('  OK Feature normalization: train stats only')
print('  OK Feature selection: train data only (|corr| with y)')
runtime_ok = total_elapsed < 1800
print(f'  {"OK" if runtime_ok else "FAIL"} Runtime {total_elapsed/60:.1f} min {"< 30 min" if runtime_ok else "> 30 min OVER BUDGET"}')
print('='*70)

if   final_r2 > 0.005: print('EXCEPTIONAL')
elif final_r2 > 0.003: print('STRONG — prize range')
elif final_r2 > 0.002: print('GOOD — above majority')
elif final_r2 > 0.001: print('MODERATE — above baseline')
elif final_r2 > 0:     print('WEAK — positive signal')
else:                   print('NO SIGNAL')

print()
print('NOTEBOOK COMPLETE v9')
if np.mean(iters_huber) < 80:
    print('WARNING: LGB_Huber avg iter < 80 — model may be over-regularized.')

## Submission Checklist (v9)

| Rule | Status |
|---|---|
| `/kaggle/input/competitions/...` exact path | PASS |
| No internet / pip install | PASS |
| CPU only | PASS |
| No cross-sample test operations | PASS |
| GroupKFold on CV_GROUP | PASS |
| Feature selection: train data only | PASS |
| Normalization: train stats only | PASS |
| Runtime guard at 25 min | PASS |
| submission.csv: 6-guard validation | PASS |